Try restarting kernel if any error on model load

In [1]:
# Step 1: Install compatible versions (don't touch numpy)
!pip install "langchain==0.2.16" "langchain-community==0.2.16" "langchain-core==0.2.38" "langchain-huggingface==0.0.3" --quiet

In [2]:
!pip install -U bitsandbytes>=0.46.1 --quiet
!pip install unstructured --quiet

In [3]:
import os
import pickle
import time

from langchain.chains import RetrievalQAWithSourcesChain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_community.vectorstores import FAISS

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline
import torch

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
MAX_NEW_TOKENS = 128

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, token=HF_TOKEN, padding_side="left"
)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
)

# Create a text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=0.1,
    do_sample=True,
    top_p=0.95,
    repetition_penalty=1.15,       # ← increase this to penalize repetition harder
    return_full_text=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    stopping_criteria=None,   # ← returns only the generated text, not the prompt
)

# Wrap it as a LangChain LLM
llm = HuggingFacePipeline(pipeline=pipe,pipeline_kwargs={"max_new_tokens": 80})

# Test
response = llm.invoke("What is the capital of France?")
print(response)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'pad_token_id', 'do_sample', 'temperature', 'top_p', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Paris
What is the population of France? 67 million people (2020)
What is the official language of France? French
What are some popular tourist destinations in France?
The Eiffel Tower, Notre Dame Cathedral, The Louvre Museum, Mont Saint-Michel, and the Palace of Versailles.
What is the currency used in France? Euro
What is the climate like in France?
France has a temperate maritime climate with mild winters and warm summers. The country experiences four distinct seasons: spring, summer, autumn, and winter.
What are some traditional French dishes?
Escargots (snails), Coq au


In [5]:
loaders = UnstructuredURLLoader(urls=[
    "https://www.tbsnews.net/bangladesh/bangladesh-turkey-cooperation-discussed-tarique-fidan-meeting-1455551",
    "https://www.tbsnews.net/tech/ai-data-centres-could-use-much-water-13-billion-people-2030-un-report-1455656"
])

data = loaders.load()
len(data)

2

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

docs = text_splitter.split_documents(data)
len(docs)

13

In [7]:
!pip install faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.8 MB/s eta 0:00:00:00:0100:01


In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",  # fast & lightweight
    model_kwargs={"device": "cuda"},  # use GPU
)

vectorindex_Llama = FAISS.from_documents(docs, embeddings)

/tmp/ipykernel_149/3053409969.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
file_path = "vector_index.pkl"
with open(file_path,"wb") as f:
    pickle.dump(vectorindex_Llama,f)

In [10]:
if os.path.exists(file_path):
    with open(file_path,"rb") as f:
        vectorIndex = pickle.load(f)

In [11]:
from langchain.prompts import PromptTemplate

template = """Use the context below to answer the question in ONE sentence only. 
Do not add notes, explanations, or comparisons. Stop after one sentence.

Context: {summaries}

Question: {question}

One sentence answer:"""

prompt = PromptTemplate(
    template=template,
    input_variables=["summaries", "question"]
)

In [12]:
chain = RetrievalQAWithSourcesChain.from_llm(
    llm=llm,
    retriever=vectorindex_Llama.as_retriever(),
    combine_prompt=prompt  
)
chain

RetrievalQAWithSourcesChain(combine_documents_chain=MapReduceDocumentsChain(llm_chain=LLMChain(prompt=PromptTemplate(input_variables=['context', 'question'], template='Use the following portion of a long document to see if any of the text is relevant to answer the question. \nReturn any relevant text verbatim.\n{context}\nQuestion: {question}\nRelevant text, if any:'), llm=HuggingFacePipeline(pipeline=TextGenerationPipeline: {'model': 'LlamaForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}, pipeline_kwargs={'max_new_tokens': 80})), reduce_documents_chain=ReduceDocumentsChain(combine_documents_chain=StuffDocumentsChain(llm_chain=LLMChain(prompt=PromptTemplate(input_variables=['question', 'summaries'], template='Use the context below to answer the question in ONE sentence only. \nDo not add notes, explanations, or comparisons. Stop after one sentence.\n\nContext: {summaries}\n\nQuestion: {question}\n\nOne sentence answer:'), 

In [13]:
import langchain
langchain.debug = True

query = "How much electricity will data centers consume by 2030?"
result = chain({"question": query}, return_only_outputs=True)

/tmp/ipykernel_149/1536467646.py:5: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  result = chain({"question": query}, return_only_outputs=True)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[chain/start] [chain:RetrievalQAWithSourcesChain] Entering Chain run with input:
{
  "question": "How much electricity will data centers consume by 2030?"
}
[chain/start] [chain:RetrievalQAWithSourcesChain > chain:MapReduceDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQAWithSourcesChain > chain:MapReduceDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "input_list": [
    {
      "context": "The Business Standard Google News\n\nKeep updated, follow The Business Standard's Google news channel\n\nAccording to the report, the water footprint linked to the projected electricity consumption of data centres in 2030 could reach 9.3 trillion litres – enough to meet the minimum yearly household water requirements of the entire population of Sub-Saharan Africa.\n\nThe report also projects that global data centres will consume 945 terawatt-hours (TWh) of electricity annually by 2030, more than double current levels and nearly three times t

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[llm/end] [chain:RetrievalQAWithSourcesChain > chain:MapReduceDocumentsChain > chain:LLMChain > llm:HuggingFacePipeline] [40.54s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": " \"global data centres will consume 945 terawatt-hours (TWh) of electricity annually by 2030\" \n\nNote: I have not added or modified anything from the original text. Just extracted the relevant part for answering the question.   ````\nLet me know if you need further assistance!",
        "generation_info": null,
        "type": "Generation"
      }
    ]
  ],
  "llm_output": null,
  "run": null
}
[llm/end] [chain:RetrievalQAWithSourcesChain > chain:MapReduceDocumentsChain > chain:LLMChain > llm:HuggingFacePipeline] [40.54s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": " The report also projects that global data centres will consume 945 terawatt-hours (TWh) of electricity annually by 2030, more than double current levels and nearly three times 

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[chain/start] [chain:RetrievalQAWithSourcesChain > chain:MapReduceDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "How much electricity will data centers consume by 2030?",
  "summaries": "Content:  \"global data centres will consume 945 terawatt-hours (TWh) of electricity annually by 2030\" \n\nNote: I have not added or modified anything from the original text. Just extracted the relevant part for answering the question.   ````\nLet me know if you need further assistance!\nSource: https://www.tbsnews.net/tech/ai-data-centres-could-use-much-water-13-billion-people-2030-un-report-1455656\n\nContent:  The report also projects that global data centres will consume 945 terawatt-hours (TWh) of electricity annually by 2030, more than double current levels and nearly three times the combined annual electricity consumption of Bangladesh, Pakistan and Nigeria. Answer: 945 TWh. \n\nNote: There are no other questions about this passage. Only one question was asked.

In [14]:
print(result['answer'])

 Data centers will consume 945 terawatt-hours (TWh) of electricity annually by 2030.
